<a href="https://colab.research.google.com/github/tanphong-sudo/voxbridge/blob/main/colab/02_build_apk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoxBridge - Notebook 02: SMOKE_VI_EN

Open this current GitHub notebook and Run all. It mounts Drive, clones source, restores
only VI/EN models, creates PhoWhisper Base / Whisper Small only when missing, runs real
JVM E2E, builds the arm64 APK and copies it to Drive with a verified checksum.

Existing OPUS/Piper models are reused. No QAI token or paid QAI jobs are needed.
The first run needs time to create the two selected ASRs; successful exports are cached on Drive.
ASR weights remain FP32. Identical decoder weights share an ONNX external-data file;
no quantization or changes to the decoder input/output contract are made.

Selected set: Silero; PhoWhisper Base (VI); Whisper Small (EN); OPUS-MT vi-en/en-vi INT8;
Piper VI/EN; VI/EN safety prompts. No ZH/KO or Whisper Tiny in this APK.
See docs/development/legacy-opus-build.md. Actual sizes are printed; a 1 GB APK is not assumed.


## 0. Mount Drive and clone or update the private source

This notebook is independent: it can run in a fresh Colab runtime without Notebook 00.


In [ ]:
import os
import pathlib
import subprocess
from google.colab import drive, userdata

DRIVE_ROOT = "/content/drive/MyDrive/VoxBridge"
REPO_URL = "https://github.com/tanphong-sudo/voxbridge.git"
REPO = pathlib.Path("/content/voxbridge")
drive.mount("/content/drive")


def run_git(args, cwd=None):
    """Run Git with the token held only in this process environment."""
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Missing GITHUB_TOKEN. In the left sidebar, open Secrets, add GITHUB_TOKEN, "
            "and enable its notebook access. The token needs read access to voxbridge."
        )

    askpass = pathlib.Path("/tmp/voxbridge-git-askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) echo x-access-token ;;\n"
        "  *) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\n"
        "esac\n"
    )
    askpass.chmod(0o700)
    env = {
        **os.environ,
        "GITHUB_TOKEN": token,
        "GIT_ASKPASS": str(askpass),
        "GIT_TERMINAL_PROMPT": "0",
    }
    try:
        return subprocess.run(["git", *args], cwd=cwd, env=env, text=True, check=True)
    finally:
        askpass.unlink(missing_ok=True)


if not (REPO / ".git").is_dir():
    if REPO.exists():
        raise RuntimeError(
            f"{REPO} exists but is not a Git checkout. Run: !rm -rf /content/voxbridge, then rerun this cell."
        )
    run_git(["clone", "--branch", "main", REPO_URL, str(REPO)])
else:
    run_git(["fetch", "origin", "main"], cwd=REPO)
    run_git(["pull", "--ff-only", "origin", "main"], cwd=REPO)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
print(f"Source ready: {REPO} @ {commit}")


## 0.5 Restore VI/EN cache and create only the missing selected ASRs


In [ ]:
import sys
import shutil

BUILD_PROFILE = "SMOKE_VI_EN"
models = pathlib.Path("/content/voxbridge-selected-models")
drive_models = pathlib.Path(DRIVE_ROOT) / "models"

def logged_run(command):
    result = subprocess.run(command, cwd=REPO, text=True)
    if result.returncode:
        raise RuntimeError(f"Command failed (exit {result.returncode}): {command[0:2]}. Read the output above.")

# Preserve the full original cache; staging is separate and contains only the selected languages.
source_models = drive_models if drive_models.is_dir() else REPO / "models"
logged_run([sys.executable, "tools/dev/model_bundle.py", str(source_models),
            "--destination", str(models), "--profile", BUILD_PROFILE, "--prepare-asr"])
check_asr = subprocess.run([sys.executable, "model-tools-python/scripts/export_selected_asr.py",
                            str(models), "--check"], cwd=REPO)
if check_asr.returncode not in (0, 1):
    raise RuntimeError("ASR cache check failed; read the output above.")
if check_asr.returncode == 1:
    # Python 3.11 isolates export tooling from Colab's Python 3.13 packages.
    venv = pathlib.Path("/content/venvs/phase2-asr")
    logged_run([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"])
    logged_run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", "--allow-existing", str(venv)])
    pip = str(venv / "bin/pip")
    logged_run([pip, "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cpu"])
    logged_run([pip, "install", "-r", "model-tools-python/requirements-phase2-asr.txt"])
    logged_run([pip, "check"])
    logged_run([str(venv / "bin/python"), "model-tools-python/scripts/export_selected_asr.py", str(models)])
logged_run([sys.executable, "tools/dev/model_bundle.py", str(models), "--profile", BUILD_PROFILE])
print("Selected Phase-2 models ready. Existing OPUS/Piper exports were preserved.")


## 1 · JDK 17 + Android SDK 34

In [ ]:
import os, subprocess
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'openjdk-17-jdk-headless', 'binutils', 'unzip', 'wget'], check=True)
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
subprocess.run(['update-alternatives', '--set', 'java',
                '/usr/lib/jvm/java-17-openjdk-amd64/bin/java'], check=True)
subprocess.run(['java', '-version'], check=True)


In [ ]:
%%bash
set -euo pipefail
set -e
SDK=/opt/android-sdk
if [ ! -x $SDK/cmdline-tools/latest/bin/sdkmanager ]; then
  mkdir -p $SDK/cmdline-tools
  wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O /tmp/clt.zip
  unzip -q /tmp/clt.zip -d $SDK/cmdline-tools
  mv $SDK/cmdline-tools/cmdline-tools $SDK/cmdline-tools/latest
fi
export ANDROID_HOME=$SDK
yes | $SDK/cmdline-tools/latest/bin/sdkmanager --licenses > /dev/null || true
$SDK/cmdline-tools/latest/bin/sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0" > /dev/null
echo "Android SDK ready"

## 2 · Gradle 8.9 + sherpa-onnx AAR

In [ ]:
%%bash
set -euo pipefail
set -e
if [ ! -x /opt/gradle/gradle-8.9/bin/gradle ]; then
  wget -q https://services.gradle.org/distributions/gradle-8.9-bin.zip -O /tmp/gradle.zip
  unzip -q /tmp/gradle.zip -d /opt/gradle
fi
/opt/gradle/gradle-8.9/bin/gradle --version

In [ ]:
# sherpa-onnx ships as prebuilt AARs in its GitHub releases (NOT on Maven).
# Use the static-ORT variant so sherpa does not contribute a second arm64 libonnxruntime.so.
# v1.13.3 regular AAR uses ORT 1.24.3, while the app's Microsoft QNN artifact is ORT 1.20.0.
# Keeping the runtimes isolated prevents libonnxruntime4j_jni.so from binding to the wrong core.
import json, pathlib, shutil, urllib.request
TAG = 'v1.13.3'
ASSET = 'sherpa-onnx-static-link-onnxruntime-1.13.3.aar'
url = f'https://github.com/k2-fsa/sherpa-onnx/releases/download/{TAG}/{ASSET}'
print('downloading', url)
libs = pathlib.Path('/content/voxbridge/android/app/libs'); libs.mkdir(parents=True, exist_ok=True)
for stale in libs.glob('sherpa-onnx-*.aar'):
    if stale.name != ASSET:
        stale.unlink()
        print('removed stale sherpa AAR:', stale.name)
dst = libs / ASSET
with urllib.request.urlopen(url, timeout=120) as resp, open(dst, 'wb') as out:
    shutil.copyfileobj(resp, out)
print('saved', dst, f'{dst.stat().st_size/1e6:.1f} MB')

# Fail closed before Gradle runs for the ABI this app actually packages. The upstream static
# AAR has an x86-only libonnxruntime.so compatibility entry; abiFilters excludes x86.
# arm64-v8a must carry sherpa JNI and must NOT carry a dynamic libonnxruntime.so.
import zipfile
TARGET_ABI = 'arm64-v8a'
with zipfile.ZipFile(dst) as z:
    names = z.namelist()
jni_libs = sorted(n for n in names if n.endswith('libsherpa-onnx-jni.so'))
ort_libs = sorted(n for n in names if n.endswith('libonnxruntime.so'))
target_jni_libs = sorted(n for n in jni_libs if n.startswith(f'jni/{TARGET_ABI}/'))
target_ort_libs = sorted(n for n in ort_libs if n.startswith(f'jni/{TARGET_ABI}/'))
assert target_jni_libs, f'AAR thieu libsherpa-onnx-jni.so cho {TARGET_ABI}: {jni_libs}'
assert not target_ort_libs, f'Static sherpa AAR unexpectedly carries {TARGET_ABI} libonnxruntime.so: {target_ort_libs}'
if ort_libs:
    print('Non-target ABI ORT entries ignored by arm64 abiFilters:', ort_libs)
print(f'AAR {TARGET_ABI} sherpa JNI uses statically linked ORT:', target_jni_libs)
print(f'OK: sherpa contributes no dynamic libonnxruntime.so for {TARGET_ABI}')


## 2.5 Stage the VI/EN APK assets and run real E2E on that exact model set


In [ ]:
assets = REPO / "android/app/src/main/assets/models"
logged_run([sys.executable, "tools/dev/model_bundle.py", str(models),
            "--destination", str(assets), "--profile", BUILD_PROFILE])

# Fail closed on ASR composition before any JVM/Gradle work. Old Drive cache entries may exist,
# but the staged APK tree must contain exactly the two selected product ASRs.
asr_root = assets / "speech_to_text"
asr_dirs = sorted(p.name for p in asr_root.iterdir() if p.is_dir())
expected_asr_dirs = ["phowhisper-base-onnx", "whisper-small-onnx"]
if asr_dirs != expected_asr_dirs:
    raise RuntimeError(f"Wrong staged ASR set: {asr_dirs}; expected exactly {expected_asr_dirs}")
for forbidden in ("whisper-tiny-onnx", "whisper-tiny-qnn", "whisper-base-onnx", "whisper-base-qnn", "whisper-small-qnn"):
    if (asr_root / forbidden).exists():
        raise RuntimeError(f"Forbidden obsolete ASR staged into APK assets: {forbidden}")
print("ASR staging locked:", asr_dirs)
# This gate tests both selected ASRs and both MT directions; failures block the build.
logged_run(["bash", "tools/e2e-jvm/run.sh", str(assets)])
print("SMOKE_VI_EN real JVM E2E passed; build may proceed.")

# Cache only the selected ASRs after the real graph gate passes. Leave the shared OPUS/Piper cache intact; obsolete ASRs are never staged into the APK.
import hashlib
import time

def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.digest()

for name in ("phowhisper-base-onnx", "whisper-small-onnx"):
    src = models / "speech_to_text" / name
    target = drive_models / "speech_to_text" / name
    target.parent.mkdir(parents=True, exist_ok=True)
    source_files = [p for p in src.rglob("*") if p.is_file()]
    if target.is_dir() and all((target / p.relative_to(src)).is_file() and
                              file_hash(p) == file_hash(target / p.relative_to(src)) for p in source_files):
        print("Drive cache already matches:", name)
        continue
    nonce = str(time.time_ns())
    temporary = target.with_name(name + ".copying-" + nonce)
    backup = target.with_name(name + ".previous-" + nonce)
    shutil.copytree(src, temporary)
    for source_file in source_files:
        copied = temporary / source_file.relative_to(src)
        if not copied.is_file() or file_hash(source_file) != file_hash(copied):
            raise RuntimeError(f"Drive ASR copy differs: {source_file.name}")
    had_previous = target.exists()
    if had_previous:
        target.rename(backup)
    try:
        temporary.rename(target)
        for source_file in source_files:
            promoted = target / source_file.relative_to(src)
            if not promoted.is_file() or file_hash(source_file) != file_hash(promoted):
                raise RuntimeError(f"Promoted Drive ASR differs: {source_file.name}")
    except Exception:
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        if backup.exists():
            backup.rename(target)
        if temporary.exists():
            shutil.rmtree(temporary)
        raise
    if backup.exists():
        shutil.rmtree(backup)
    print("Verified ASR cached on Drive:", target)

# The selected pair has passed real JVM E2E. Remove superseded ASR cache directories so a
# future restore cannot resurrect them. The names are exact and deliberately narrow.
for obsolete in ("whisper-tiny-onnx", "whisper-tiny-qnn", "whisper-base-onnx", "whisper-base-qnn", "whisper-small-qnn"):
    stale = drive_models / "speech_to_text" / obsolete
    if stale.is_dir():
        shutil.rmtree(stale)
        print("Removed obsolete Drive ASR cache:", stale)


## 3 · Build

In [ ]:
%%bash
set -euo pipefail
cd /content/voxbridge
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
export ANDROID_HOME=/opt/android-sdk
LOG=/content/voxbridge-gradle-build.log

# Print the resolved runtime graph before building so ONNX Runtime contributors are auditable.
/opt/gradle/gradle-8.9/bin/gradle :android:app:dependencies --configuration debugRuntimeClasspath --no-daemon
/opt/gradle/gradle-8.9/bin/gradle :android:app:dependencyInsight --dependency onnxruntime --configuration debugRuntimeClasspath --no-daemon

# Keep Gradle diagnostics visible in Colab and retain them for a failed build.
if ! /opt/gradle/gradle-8.9/bin/gradle :android:app:assembleDebug --no-daemon --stacktrace 2>&1 | tee "$LOG"; then
  echo "BUILD FAILED - full Gradle log: $LOG" >&2
  exit 1
fi
ls -la android/app/build/outputs/apk/debug/


## 4 · Verify the NO-INTERNET guarantee + ship the APK

In [ ]:
%%bash
set -euo pipefail
APK=/content/voxbridge/android/app/build/outputs/apk/debug/app-debug.apk
BUILD_TOOLS=/opt/android-sdk/build-tools/34.0.0
AAPT=$BUILD_TOOLS/aapt2
APKSIGNER=$BUILD_TOOLS/apksigner
test -s "$APK"
MAX_APK_BYTES=2147483647
APK_BYTES="$(wc -c < "$APK" | tr -d '[:space:]')"
printf 'APK size: %s bytes\n' "$APK_BYTES"
[[ "$APK_BYTES" -lt "$MAX_APK_BYTES" ]] || { echo "FAIL: APK exceeds the 2 GiB install-safety limit"; exit 1; }
unzip -t "$APK" >/dev/null
"$APKSIGNER" verify --verbose --print-certs "$APK"
BADGING="$("$AAPT" dump badging "$APK")"
printf '%s\n' "$BADGING" | grep -E "^package|uses-permission|launchable-activity"
if [[ "$BADGING" == *android.permission.INTERNET* ]]; then
  echo "FAIL: APK declares INTERNET"; exit 1
fi
if [[ "$BADGING" != *"com.voxbridge.ui.MainActivity"* ]]; then
  echo "FAIL: expected launcher missing"; exit 1
fi

# APK model gate: exact ASR allowlist, independent of the staging check above.
ASR_DIRS="$(unzip -Z1 "$APK" | sed -n 's#^assets/models/speech_to_text/\([^/]*\)/.*#\1#p' | sort -u)"
EXPECTED_ASR_DIRS="$(printf '%s\n' phowhisper-base-onnx whisper-small-onnx | sort)"
printf 'Packaged ASR directories:\n%s\n' "$ASR_DIRS"
[[ "$ASR_DIRS" == "$EXPECTED_ASR_DIRS" ]] || {
  echo "FAIL: APK ASR set is not exactly PhoWhisper Base + Whisper Small"; exit 1;
}
if unzip -Z1 "$APK" | grep -Eq 'assets/models/speech_to_text/(whisper-tiny|whisper-base)(-|/)'; then
  echo "FAIL: APK contains obsolete Whisper Tiny/Base assets"; exit 1
fi

# Native ORT gate: there must be exactly one Microsoft 1.20 core/JNI pair, while sherpa's ORT
# stays statically linked inside libsherpa-onnx-jni.so. Do not use pickFirst to hide collisions.
NATIVE_LIST="$(unzip -Z1 "$APK" | grep -E '^lib/arm64-v8a/.*\.so$' || true)"
printf '%s\n' "$NATIVE_LIST" | grep -E 'lib/arm64-v8a/(libonnxruntime|libsherpa-onnx).*\.so$' || true
CORE_COUNT="$(printf '%s\n' "$NATIVE_LIST" | grep -c '^lib/arm64-v8a/libonnxruntime.so$' || true)"
JNI_COUNT="$(printf '%s\n' "$NATIVE_LIST" | grep -c '^lib/arm64-v8a/libonnxruntime4j_jni.so$' || true)"
SHERPA_COUNT="$(printf '%s\n' "$NATIVE_LIST" | grep -c '^lib/arm64-v8a/libsherpa-onnx-jni.so$' || true)"
[[ "$CORE_COUNT" == 1 ]] || { echo "FAIL: expected exactly one libonnxruntime.so, got $CORE_COUNT"; exit 1; }
[[ "$JNI_COUNT" == 1 ]] || { echo "FAIL: expected exactly one libonnxruntime4j_jni.so, got $JNI_COUNT"; exit 1; }
[[ "$SHERPA_COUNT" == 1 ]] || { echo "FAIL: expected exactly one libsherpa-onnx-jni.so, got $SHERPA_COUNT"; exit 1; }

TMP="$(mktemp -d)"
trap 'rm -rf "$TMP"' EXIT
unzip -p "$APK" lib/arm64-v8a/libonnxruntime.so > "$TMP/libonnxruntime.so"
unzip -p "$APK" lib/arm64-v8a/libonnxruntime4j_jni.so > "$TMP/libonnxruntime4j_jni.so"
unzip -p "$APK" lib/arm64-v8a/libsherpa-onnx-jni.so > "$TMP/libsherpa-onnx-jni.so"
CORE_SYM="$(readelf --wide -Ws "$TMP/libonnxruntime.so" | awk '/OrtGetApiBase/{print $8; exit}')"
JNI_SYM="$(readelf --wide -Ws "$TMP/libonnxruntime4j_jni.so" | awk '/OrtGetApiBase/{print $8; exit}')"
CORE_VER="${CORE_SYM##*@}"
JNI_VER="${JNI_SYM##*@}"
printf 'ORT core symbol: %s\nORT Java JNI requirement: %s\n' "$CORE_SYM" "$JNI_SYM"
[[ "$CORE_VER" == "VERS_1.20.0" ]] || { echo "FAIL: packaged ORT core is not 1.20.0"; exit 1; }
[[ "$JNI_VER" == "$CORE_VER" ]] || { echo "FAIL: ORT JNI/core symbol versions differ: $JNI_VER vs $CORE_VER"; exit 1; }
if readelf -d "$TMP/libsherpa-onnx-jni.so" | grep -q 'Shared library: \[libonnxruntime.so\]'; then
  echo "FAIL: sherpa JNI still dynamically depends on libonnxruntime.so"; exit 1
fi
if readelf --wide -Ws "$TMP/libsherpa-onnx-jni.so" | grep -E 'UND.*OrtGetApiBase' >/dev/null; then
  echo "FAIL: sherpa JNI still imports OrtGetApiBase dynamically"; exit 1
fi

echo "OK: packaged dynamic ONNX Runtime = 1.20.0; sherpa ORT is isolated by static linking"
sha256sum "$APK"
echo "OK: signed APK has a valid launcher, no INTERNET permission, and a compatible ORT native pair"


In [ ]:
# Inspect this build; no old Drive artifact or adb device is required.
apk = REPO / "android/app/build/outputs/apk/debug/app-debug.apk"
logged_run(["bash", "tools/dev/inspect-apk.sh", str(apk), "/opt/android-sdk/build-tools/34.0.0"])


In [ ]:
import hashlib
import pathlib
import shutil
import time
from google.colab import drive

apk = pathlib.Path("/content/voxbridge/android/app/build/outputs/apk/debug/app-debug.apk")
assert apk.is_file(), f"app-debug.apk does not exist: {apk}"
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

digest = sha256_file(apk)
name = f"voxbridge-debug-SMOKE_VI_EN-{time.strftime('%Y%m%d-%H%M')}.apk"
drive.mount("/content/drive")
dst = pathlib.Path("/content/drive/MyDrive/VoxBridge/artifacts/apk")
dst.mkdir(parents=True, exist_ok=True)
out = dst / name
shutil.copy2(apk, out)
if sha256_file(out) != digest:
    raise RuntimeError("Drive copy checksum mismatch; do not install this copy.")
(out.with_suffix(".apk.sha256")).write_text(f"{digest}  {name}\n")
print("APK:", out, f"{out.stat().st_size / 1e6:.1f} MB")
print("SHA-256:", digest)
print("Drive checksum:", out.with_suffix(".apk.sha256"))
print("Install note: arm64, Android 10+. Model extraction also needs free storage. Test both directions in airplane mode.")
